<a href="https://colab.research.google.com/github/Farhana-Mahbuba-Laboni/CSE499A/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Version 1**
**Project Design:**
We are creating a simple AI assistant that can explain and solve basic Bengali math problems. To achieve this, we will fine-tune a pre-trained language model to understand and generate responses in Bengali. The model will be fine-tuned on a dataset of math problems and their solutions written in Bengali. The goal is to develop an interactive assistant that can help users with their math queries in Bengali.

**Tools Used:**
We are using the Qwen2.5-7B-Instruct model released by Alibaba. To improve computational efficiency, we are using an optimized version of the model from Unsloth. Specifically, we are using the 4-bit quantized version of the model with Low-Rank Adaptation (LoRA), which allows us to fine-tune the model with fewer parameters while maintaining performance. Additionally, we are applying the Parameter-Efficient Fine-Tuning (PEFT) technique to further reduce the number of trainable parameters. We are using the Hugging Face library to load the dataset. The framework is based on PyTorch, and we are using the Hugging Face Trainer API for training.

**Work Plan:**
We will fine-tune the model on the dataset of Bengali math problems and their solutions. The fine-tuned model will be used to create an API backend using FastAPI. This API will handle user input and return the model’s response. A simple frontend will be built using either Streamlit or Next.js to interact with the API. The frontend will take user input, send it to the API, and display the model’s response.

## **Version 2**
**Project Title:** Bengali AI Math Assistant

**Project Overview:**  
We are developing a simple AI assistant capable of understanding and solving basic math problems in Bengali. The assistant will be fine-tuned on a dataset of Bengali math problems and their solutions, enabling it to provide helpful and accurate explanations in Bengali. The aim is to build an interactive tool that assists users in solving math queries in their native language.

**Tools & Technologies Used:**

- **Model:** Qwen2.5-7B-Instruct by Alibaba  
- **Optimization:**  
  - Using the optimized version from Unsloth  
  - 4-bit quantization for faster and memory-efficient training/inference  
  - Low-Rank Adaptation (LoRA) for efficient fine-tuning  
  - Parameter-Efficient Fine-Tuning (PEFT) for reducing trainable parameters  
- **Libraries:**  
  - Hugging Face Transformers and Datasets  
  - PyTorch  
  - PEFT & BitsAndBytes for quantized model support  
- **API Framework:** FastAPI  
- **Frontend:** Streamlit or Next.js (for user interaction)

**Work Plan:**

1. **Fine-Tuning**  
   - Fine-tune the Qwen2.5-7B-Instruct model using the Bengali math dataset (problem-solution pairs).  
   - Apply LoRA and PEFT to minimize resource usage while maintaining performance.

2. **API Development**  
   - Build a FastAPI backend to serve the fine-tuned model.  
   - The API will accept user queries and return the model-generated responses.

3. **Frontend Development**  
   - Create a simple frontend using Streamlit or Next.js.  
   - Users can input math questions, and the interface will display the assistant’s responses in Bengali.

#### **Import the required libraries**


In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kq6840ns/unsloth_ec60e5cedbce474c9abff05132bfa307
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kq6840ns/unsloth_ec60e5cedbce474c9abff05132bfa307
  Resolved https://github.com/unslothai/unsloth.git to commit 6c234d5a66adb76b9b93fb0f2445648199d88e66
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!


`from unsloth import FastLanguageModel, is_bfloat16_supported`  
This comes from the [Unsloth](https://github.com/unslothai/unsloth) library, which is designed to fine-tune large language models quickly and with low memory usage.

- `FastLanguageModel` is a wrapper around Hugging Face models that supports optimizations like 4-bit quantization and adapter-based methods (like LoRA) for fast, efficient fine-tuning on consumer hardware.
- `is_bfloat16_supported` is a utility function that checks whether the current hardware (e.g., your GPU) supports the `bfloat16` data type. This is useful for deciding whether to use `bfloat16`, `float16`, or `float32` during training or inference.

`import torch`  
This imports PyTorch, a popular machine learning framework that provides tools for tensor operations, GPU acceleration, and building deep learning models. It is the core engine behind model computation, gradient updates, and tensor manipulation.

`from datasets import load_dataset`  
This function comes from the Hugging Face Datasets library. It allows you to easily load public datasets from the Hugging Face Hub or your own datasets (like CSV or JSON files). It handles downloading, caching, and preparing datasets for use in training and evaluation.

`from trl import SFTTrainer`  
This is part of the TRL (Transformers Reinforcement Learning) library by Hugging Face, although it's commonly used for supervised fine-tuning as well.

- `SFTTrainer` stands for **Supervised Fine-Tuning Trainer**, which provides an easy interface for training language models on instruction datasets or any supervised objective. It integrates well with Hugging Face models and supports techniques like LoRA and PEFT (Parameter-Efficient Fine-Tuning).

`from transformers import TrainingArguments`  
This is part of the Transformers library by Hugging Face.

- `TrainingArguments` is used to configure the training process, including key hyperparameters such as learning rate, batch size, number of training epochs, output directory, evaluation strategy, precision settings (like `fp16` or `bf16`), and logging behavior.


#### **Get the model and tokenizer**


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
    rope_scaling=True,
)

==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.16G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

- **Load the model and tokenizer**

  - This single function loads both the model and the tokenizer together—optimized for low-resource environments.
  - It wraps Hugging Face’s model-loading capabilities with additional speed and memory optimizations from Unsloth.

- **model_name="unsloth/Qwen2.5-7B-Instruct"**

  - Specifies the pre-trained instruction-tuned Qwen 2.5 7B model hosted by Unsloth.
  - This version is specially optimized by Unsloth for faster loading and fine-tuning with low GPU memory.
  - It's a great choice for Bangla instruction-based tasks due to its multilingual capabilities and instruction-tuning.

- **max_seq_length=4096**

  - Sets the maximum input sequence length to 4096 tokens.
  - By default, most models support 2048 tokens, but this setting allows handling much longer text passages—essential for document-level tasks or longer conversations.

- **dtype=None**

  - Lets Unsloth automatically determine the best precision type based on your GPU capabilities.
  - If left as `None`, it may choose `bfloat16`, `float16`, or `float32` depending on what's supported.

- **load_in_4bit=True**

  - Loads the model in 4-bit precision using `bitsandbytes`, drastically reducing VRAM usage.
  - This is ideal for consumer GPUs with 8GB–16GB memory.
  - Helps enable training or inference of large models like Qwen-7B on mid-range GPUs.

- **# rope_scaling=True**
  - If enabled, it would apply RoPE (Rotary Positional Embedding) scaling, which allows models to generalize better to longer context lengths.
  - Useful for training or inference beyond the original sequence length the model was trained on.


#### **Enable PEFT (Parameter-Efficient Fine-Tuning)**


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,# rank of the low-rank decomposition. Can change this to 8, 16, 32, 64 etc.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
    lora_alpha = 16,
    lora_dropout = 0., # support any but 0 is optimized
    bias = "none", # support any, but "none" is optimized
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None
)

Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


We're enabling **parameter-efficient fine-tuning (PEFT)** using **LoRA (Low-Rank Adaptation)** on the model, with Unsloth’s optimized `get_peft_model()` method. This lets us fine-tune large models like Qwen 2.5-7B with very little GPU memory while still achieving good performance.

### Arguments for `get_peft_model()`:

- **`model`**

  - This is the already loaded Qwen 2.5 7B model from `FastLanguageModel.from_pretrained()`.
  - We pass it here to apply LoRA layers for efficient fine-tuning.

- **`r = 16`**

  - This sets the **rank** of the low-rank decomposition used in LoRA.
  - Common values are 8, 16, 32, 64 — higher values mean better capacity but more memory usage.
  - `r=16` is a good trade-off between performance and memory.

- **`target_modules`**

  - A list of transformer modules where LoRA adapters will be inserted.
  - Includes attention projection layers like:
    - `"q_proj"`, `"k_proj"`, `"v_proj"`, `"o_proj"` — query, key, value, and output projections.
    - `"gate_proj"`, `"down_proj"`, `"up_proj"` — parts of the MLP (feedforward) block.
  - This selection ensures we cover key components responsible for model understanding and generation.

- **`lora_alpha = 16`**

  - The scaling factor for the LoRA weights.
  - This is typically the same as `r` (rank) or slightly higher. It balances training stability and adaptation capacity.

- **`lora_dropout = 0.`**

  - Dropout applied to LoRA layers during training.
  - `0.0` means no dropout, which is optimal for stability and performance in most use cases.

- **`bias = "none"`**

  - Whether to fine-tune bias terms in the model.
  - `"none"` means only LoRA layers are trainable—more memory-efficient.
  - Other options like `"all"` allow bias training, but consume more memory.

- **`use_gradient_checkpointing = "unsloth"`**

  - Enables Unsloth’s efficient gradient checkpointing.
  - This technique reduces memory usage by re-computing intermediate activations during backpropagation.
  - Great for fitting larger models on limited VRAM.

- **`random_state = 3407`**

  - Sets the seed for random number generators to ensure reproducibility of training runs.

- **`use_rslora = False`**

  - Disables **rank-stabilized LoRA**, an experimental technique.
  - Setting this to `False` ensures standard LoRA is used (faster and more stable in most cases).

- **`loftq_config = None`**
  - Placeholder for using **LoFTQ** (LoRA + quantization-aware training), an advanced PEFT technique.
  - We set it to `None` here since we're not using quantization-aware fine-tuning.


#### **Load the dataset**


In [ ]:
dataset = load_dataset("hamim-87/Ashrafur_bangla_math", split="train")

# Clean up the dataset
print(f"Original dataset size: {len(dataset)}")
dataset = dataset.filter(lambda example: example['problem'] is not None and example['solution'] is not None)
print(f"Filtered dataset size: {len(dataset)}")

# Split the dataset into train and test sets
split_dataset = dataset.train_test_split(test_size=0.001, seed=42)
train_dataset = split_dataset['train']
test_dataset = split_dataset['test']

numina-math-cot-bn.csv:   0%|          | 0.00/2.19G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/859323 [00:00<?, ? examples/s]

Original dataset size: 859323


Filter:   0%|          | 0/859323 [00:00<?, ? examples/s]

Filtered dataset size: 859319


We're preparing a Bangla Math dataset for training and evaluation using Hugging Face’s `datasets` library.

- **Dataset Loading**

  - `hamim-87/Ashrafur_bangla_math` is a dataset hosted on Hugging Face 🤗 Hub.
  - We’re loading only the `train` split since it contains the full set of examples.
  - The dataset includes Bangla math problems (`"problem"`) and their corresponding solutions (`"solution"`), useful for Bangla-language instruction tuning.

- **Data Cleaning**

  - After loading, we print the original dataset size.
  - We then filter out any examples where the problem or solution is `None`.
  - This step ensures we don’t train or evaluate on incomplete or empty entries that could harm model performance or cause errors.
  - After filtering, we print the updated size to see how many valid examples are left.

- **Train/Test Splitting**

  - We use `train_test_split` to divide the cleaned dataset:
    - 90% goes to `train_dataset`
    - 10% goes to `test_dataset`
  - `seed=42` ensures consistent splits across runs.
  - This is important for evaluating the model later on unseen examples without leaking information from training.

- **Final Result**
  - You now have a cleaned, filtered, and split dataset.
  - It’s perfectly formatted for training Bangla math reasoning models like Qwen or LLaMA Instruct.

Conceptually, your dataset will be structured like this:

| instruction                  | output           |
| ---------------------------- | ---------------- |
| "একটি বাক্সে ৫টি কলম আছে..." | "৫ × ৩ = ১৫ কলম" |


#### **Preprocess the dataset**


In [ ]:
def prepare_train_prompt(example):
    messages = [
        {
            "role": "system",
            "content": "You are a highly skilled math assistant. You can solve any mathematical problem in Bangla."
        },
        {
            "role": "user",
            "content": example['problem']
        },
        {
            "role": "assistant",
            "content": example['solution']
        }
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted_text}

train_dataset = train_dataset.map(prepare_train_prompt)

Map:   0%|          | 0/858459 [00:00<?, ? examples/s]

**We're formatting the dataset examples into chat-style conversations for training chat-based models (like Qwen or ChatGPT).**

- **Define Chat Message Format**

  - We simulate a full chat using a list of `messages`, each with a `role` and `content`.
  - This time, we include a **system message**, which sets the assistant's identity and behavior.

  - `role: "system"` — Describes the assistant as a Bangla math expert.
  - `role: "user"` — Contains the actual math problem in Bangla (`example['problem']`).
  - `role: "assistant"` — Contains the solution in Bangla (`example['solution']`).

- **Apply Chat Template**

  - `tokenizer.apply_chat_template(...)` transforms the message list into one long formatted prompt:
    - `tokenize=False`: Keeps the output as a plain string.
    - `add_generation_prompt=False`: Doesn’t add an extra assistant prompt at the end.
  - This formatting is critical for models like **Qwen** or **ChatGPT**, which were trained on data using similar chat formatting.

- **Return Formatted Prompt**

  - The function returns a dictionary: `{"text": formatted_text}`
  - This makes it compatible with Hugging Face dataset mapping.

- **Apply Function to Dataset**
  - `train_dataset.map(prepare_train_prompt)` applies the formatting to all samples in the dataset.
  - Each example gets a new `text` field containing the full chat-formatted prompt.

### **Final Result (Example):**

Before:

| problem                       | solution          |
| ----------------------------- | ----------------- |
| "৫টি কলম প্রতিটি ৩ টাকায়..." | "৫ × ৩ = ১৫ টাকা" |

After:

| problem                     | solution        | text                                                                                                                                                                                                                                                                  |
| --------------------------- | --------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| ৫টি কলম প্রতিটি ৩ টাকায়... | ৫ × ৩ = ১৫ টাকা | <\|im_start\|>system<br>You are a highly skilled math assistant. You can solve any mathematical problem in Bangla.<br><\|im_end\|><br><\|im_start\|>user<br>৫টি কলম প্রতিটি ৩ টাকায়...<br><\|im_end\|><br><\|im_start\|>assistant<br>৫ × ৩ = ১৫ টাকা<br><\|im_end\|> |


#### **Finetune the model**


In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 2, # Set this for 1 full training run.
        max_steps = 25,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        save_strategy= "steps",
        logging_steps = 10,
        save_steps = 10,
        optim = "paged_adamw_32bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/858459 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 858,459 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176/7,000,000,000 (0.58% trained)


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.50 GiB. GPU 0 has a total capacity of 14.74 GiB of which 228.12 MiB is free. Process 35129 has 14.51 GiB memory in use. Of the allocated memory 14.15 GiB is allocated by PyTorch, and 216.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

- **Initialize `SFTTrainer`**

  - `model`: The LoRA-injected, quantized Qwen model.
  - `tokenizer`: Matches the model and supports eos token, padding, etc.
  - `train_dataset`: Preprocessed dataset with `"text"` field containing instruction-style prompts.
  - `dataset_text_field = "text"`: Tells the trainer which field in the dataset contains the input text for training.
  - `max_seq_length = 4096`: Truncate/clip inputs at 4096 tokens max.
  - `dataset_num_proc = 2`: Number of CPU processes to use for data preprocessing.
  - `packing = False`: Don’t pack multiple prompts into a single sequence (avoids overlap between examples).

- **TrainingArguments**

  - `per_device_train_batch_size = 2`: Each GPU processes 2 examples per step.
  - `gradient_accumulation_steps = 4`: Gradients are accumulated over 4 steps → effective batch size = 2×4 = 8.
  - `warmup_steps = 5`: Gradually ramp up learning rate over first 5 steps to stabilize training.
  - `num_train_epochs = 1`: Run for one full pass over the training data.
  - `max_steps = 25`: Override to stop training after 25 steps (commented out).
  - `learning_rate = 2e-4`: Relatively high LR for LoRA fine-tuning (often okay for few epochs).
  - `fp16 = not is_bfloat16_supported()`: Use FP16 if BF16 is not available.
  - `bf16 = is_bfloat16_supported()`: Use BF16 on modern GPUs (like A100) for faster training and less memory.
  - `logging_steps = 10`: Log loss and metrics after every 10 step.
  - `save_steps = 10`: Save model checkpoint every 10 steps.
  - `optim = "paged_adamw_8bit"`: Use 8-bit AdamW optimizer for lower memory usage.
  - `weight_decay = 0.01`: Regularization to avoid overfitting.
  - `lr_scheduler_type = "cosine"`: Use cosine decay for learning rate.
  - `seed = 3407`: For reproducibility.
  - `output_dir = "outputs"`: Save model and logs to this folder.
  - `report_to = "none"`: Disable external loggers (like W&B, TensorBoard). Use `"wandb"` to enable logging.

- **Training**
  - `trainer.train()` starts the training loop.
  - It applies the LoRA adapters, optimizer, learning rate scheduler, and loss computation for instruction-tuning the model on Bangla math prompts.


#### **Save the model**


In [ ]:
model.save_pretrained("Qwen2.5-7B-Instruct-bangla-math")
trainer.save_model("Qwen2.5-7B-Instruct-bangla-math-trainer")

- **Model Saving**
  - `model.save_pretrained`: This is a method from Hugging Face's `transformers` library used to save the model and its configuration.
  - `"Qwen2.5-7B-Instruct-bangla-math"`: The directory name where the model will be saved. The model weights, configuration files, and any necessary tokenizer files will be stored in this folder.
  - The previous method doesn't save the whole model, but only the LoRA weights. To save the full model, we need to call `trainer.save_model()` as well.


#### **Evaluate the model**


In [ ]:
def prepare_inference_prompt(example):
    messages = [
        {
            "role": "system",
            "content": "You are a highly skilled math assistant. You can solve any mathematical problem in Bangla."
        },
        {
            "role": "user",
            "content": example['problem']
        }
    ]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return {"text": formatted_text, "reference_solution": example['solution']}

inference_test_set = test_dataset.map(prepare_inference_prompt, remove_columns=test_dataset.column_names)

Map:   0%|          | 0/860 [00:00<?, ? examples/s]

**Similar to the training process, we need to format the test dataset into a chat-style conversation.**

- **Define Chat Message Format**

  - We simulate a full chat using a list of `messages`, each with a `role` and `content`.
  - This time, we include a **system message**, which sets the assistant's identity and behavior.

  - `role: "system"` — Describes the assistant as a Bangla math expert.
  - `role: "user"` — Contains the actual math problem in Bangla (`example['problem']`).
  - We are not including the `role: "assistant"` message here, as we want the model to generate the answer.

- **Apply Chat Template**

  - `tokenizer.apply_chat_template(...)` transforms the message list into one long formatted prompt:
    - `tokenize=False`: Keeps the output as a plain string.
    - `add_generation_prompt=True`: Adds an extra assistant prompt at the end, which is useful for generating the model's response.
  - This formatting is critical for models like **Qwen** or **ChatGPT**, which were trained on data using similar chat formatting.

- **Return Formatted Prompt**

  - The function returns a dictionary: `{"text": formatted_text, "reference_solution": example['solution']}`
  - The `reference_solution` field contains the expected answer for evaluation purposes.

- **Apply Function to Dataset**
  - `test_dataset.map(prepare_inference_prompt)` applies the formatting to all samples in the dataset.
  - Each example gets a new `text` field containing the full chat-formatted prompt, and a `reference_solution` field containing the expected answer.

### **Final Result (Example):**

Before:

| problem                       | solution          |
| ----------------------------- | ----------------- |
| "৫টি কলম প্রতিটি ৩ টাকায়..." | "৫ × ৩ = ১৫ টাকা" |

After:

| problem                     | solution        | text                                                                                                                                                                                                                                               |
| --------------------------- | --------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| ৫টি কলম প্রতিটি ৩ টাকায়... | ৫ × ৩ = ১৫ টাকা | <\|im_start\|>system<br>You are a highly skilled math assistant. You can solve any mathematical problem in Bangla.<br><\|im_end\|><br><\|im_start\|>user<br>৫টি কলম প্রতিটি ৩ টাকায়...<br><\|im_end\|><br><\|im_start\|>assistant<br><\|im_end\|> |


In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
    inference_test_set[0]['text'], return_tensors="pt"
).to('cuda')

outputs = model.generate(**inputs, max_new_tokens=4096, use_cache=True)
input_length = inputs['input_ids'].shape[1]
generated_ids = outputs[0, input_length:]
generated_solution = tokenizer.decode(generated_ids, skip_special_tokens=True)


print(f"\n🔍 Prompt:\n{inference_test_set[0]['text']}")
print(f"\n💡 Prediction:\n{generated_solution}")
print(f"\n✅ Ground Truth:\n{inference_test_set[0]['reference_solution']}")


🔍 Prompt:
<|im_start|>system
You are a highly skilled math assistant. You can solve any mathematical problem in Bangla.<|im_end|>
<|im_start|>user
একটি 100 লিটার মিশ্রণে দুধ ও জলের অনুপাত 3 : 2. 48 লিটার জল যোগ করার পর মিশ্রণে দুধ ও জলের নতুন অনুপাত কী হবে?<|im_end|>
<|im_start|>assistant


💡 Prediction:
আপনার বাংলায় প্রশ্নটি সম্পূন করে আমি বাংলায় উত্তর দিব।

একটি 100 লিটর মিশ্রণে দুধ ও জলের অনুপাত 3:2। 8 লিটর জাগরণ্ত যোগ করার পর মিশ্রণে দুধ ও জলের নতুন অনুপাত কী হবে?

সমাধান:

আমরা প্রথমে মিশ্রণের আদি দুধ ও জলের অনুপাত নির্ণয় করব।

মিশ্রণের আদি দুধ ও জলের অনুপাত 3:2। মিশ্রণের আদি আয়তন 100 লিটর। এখন দুধ ও জলের অনুপাত নির্ণয় করি:

দুধ ও জলের আদি অনুপাত = 3:2

এই অনুপাতে মিশ্রণের আদি আয়তন 100 লিটর। তাহলে,

দুধের আদি আয়তন = (3 / (3 + 2)) × 100 = (3 / 5) × 100 = 60 লিটর

জলের আদি আয়তন = (2 / (3 + 2)) × 100 = (2 / 5) × 100 = 40 লিটর

এখন, 8 লিটর জাগরণ্ত যোগ করার পর মিশ্রণে দুধ ও জলের অনুপাত নির্ণয় করি।

মিশ্রণের আয়তন = 100 + 8 = 108 লিটর

জাগরণ্তের আয়তন = 8 লিটর

তাহলে, নতুন মিশ

**Inference Setup**

- **`FastLanguageModel.for_inference(model)`**  
  Prepares the model for inference, optimizing it for prediction tasks. This ensures the model is in the correct configuration for generating outputs.

- **Tokenization**  
  The prompt from the inference dataset is tokenized using the tokenizer. The tokenized input is converted to PyTorch tensors and moved to the GPU using `.to('cuda')` for efficient processing.

- **Generating Output**

  - The model generates text based on the input tokens.
  - `max_new_tokens=4096` allows the model to produce a long response if needed.
  - `use_cache=True` enables reuse of key/value attention states during decoding, which speeds up the generation process.

**Extracting the Generated Response**

- The generated output includes both the prompt and the new tokens.
- To isolate the model's answer, only the tokens after the input prompt are extracted.

**Decoding the Output**

- The extracted token IDs are decoded back into readable Bangla text.
- Special tokens (like padding or end-of-sequence markers) are skipped to clean up the final output.

- **Comparison with Ground Truth**

  - The original prompt, the model’s generated solution, and the ground truth solution from the dataset are displayed.
  - This helps compare the prediction with the actual answer for evaluation.
